In [ ]:
!pip install -q "pyarrow<24,>=19.0.0" transformers datasets accelerate bitsandbytes scikit-learn
import os, re, time
from collections import Counter
import numpy as np, pandas as pd, torch
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CSV_PATH = "/content/drive/MyDrive/valid.csv"
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
SEED, MAX_LEN = 42, 512
torch.manual_seed(SEED); np.random.seed(SEED)

# 1. Exact Same Table 1 Filtered Multi-Sense Ambiguous Split (SEED = 42)
df = pd.read_csv(CSV_PATH, nrows=300000).dropna(subset=["TEXT", "LOCATION", "LABEL"])
df["LOCATION"] = df["LOCATION"].astype(int)
df["ACRONYM"] = [w[l] if 0 <= l < len(w) else "" for w, l in zip(df["TEXT"].str.split(), df["LOCATION"])]
df = df[df["ACRONYM"] != ""]

ambig = df[df["ACRONYM"].isin(df.groupby("ACRONYM")["LABEL"].nunique()[lambda x: x >= 2].index)]
ambig = ambig[ambig["LABEL"].isin(ambig["LABEL"].value_counts()[lambda x: x >= 30].index)]
top40 = ambig.groupby("ACRONYM").filter(lambda g: g["LABEL"].nunique() >= 2)["ACRONYM"].value_counts().head(40).index.tolist()

np.random.shuffle(top40)
seen_pool = ambig[ambig["ACRONYM"].isin(top40[:30])]
unseen_pool = ambig[ambig["ACRONYM"].isin(top40[30:])]

sampled_seen, _ = train_test_split(seen_pool, train_size=1500, stratify=seen_pool["LABEL"], random_state=SEED)
train_df, test_seen_df = train_test_split(sampled_seen, train_size=1000, test_size=500, stratify=sampled_seen["LABEL"], random_state=SEED)
test_unseen_df = unseen_pool.sample(n=min(500, len(unseen_pool)), random_state=SEED)

# 2. Load Un-Tuned Base Model (4-Bit NF4, Same Quantization & Precision as LoRA Runs)
SYS = "You are a clinical and biomedical text parser. Expand the bracketed abbreviation based on the context. Output ONLY the exact canonical expansion string in lowercase, with no punctuation, quotes, or explanation."
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.pad_token or tok.eos_token
tok.padding_side = "left"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
model.eval()

def extract_ctx(row, win=60):
    w, loc = str(row["TEXT"]).split(), int(row["LOCATION"])
    abbr = w[loc] if 0 <= loc < len(w) else row["ACRONYM"]
    if 0 <= loc < len(w): w[loc] = f"[{abbr}]"
    s, e = max(0, loc - win), min(len(w), loc + win + 1)
    return " ".join(w[s:e]), abbr

# Pick 3 fixed in-context examples from train_df for Three-Shot prompting
few_shot_rows = train_df.sample(n=3, random_state=SEED)
FEW_SHOT_MSGS = []
for _, fs_r in few_shot_rows.iterrows():
    fs_ctx, fs_abbr = extract_ctx(fs_r, win=30)
    FEW_SHOT_MSGS.append({"role": "user", "content": f"Abbreviation: [{fs_abbr}]\nContext: {fs_ctx}\nExpansion:"})
    FEW_SHOT_MSGS.append({"role": "assistant", "content": str(fs_r["LABEL"]).strip().lower()})

def build_prompt(row, three_shot=False):
    ctx, abbr = extract_ctx(row, win=60)
    msgs = [{"role": "system", "content": SYS}]
    if three_shot:
        msgs.extend(FEW_SHOT_MSGS)
    msgs.append({"role": "user", "content": f"Abbreviation: [{abbr}]\nContext: {ctx}\nExpansion:"})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

# 3. Evaluation Loop (Strict Greedy Decoding, Identical Normalization & Metrics)
norm = lambda s: re.sub(r"\s+", " ", re.sub(r"[^\w\s-]", "", str(s).lower())).strip()

def tok_f1(p, t):
    pt, tt = p.split(), t.split()
    if not pt or not tt: return float(pt == tt)
    c = sum((Counter(pt) & Counter(tt)).values())
    return (2 * c / (len(pt) + len(tt))) if c else 0.0

def evaluate(df_split, three_shot=False, bs=32):
    prompts = [build_prompt(r, three_shot=three_shot) for _, r in df_split.iterrows()]
    preds, t0 = [], time.time()
    with torch.no_grad():
        for i in range(0, len(prompts), bs):
            inp = tok(prompts[i:i+bs], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(model.device)
            prompt_len = inp["input_ids"].shape[1]
            out = model.generate(**inp, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
            preds.extend(norm(tok.decode(seq[prompt_len:], skip_special_tokens=True)) for seq in out)
    elapsed = time.time() - t0
    targs = [norm(l) for l in df_split["LABEL"]]
    em = accuracy_score(targs, preds)
    mf1 = f1_score(targs, preds, average="macro", zero_division=0)
    tf1 = np.mean([tok_f1(p, t) for p, t in zip(preds, targs)])
    return em, mf1, tf1, elapsed

torch.cuda.reset_peak_memory_stats()

for mode_name, is_3shot in [("Zero-Shot (No LoRA)", False), ("Three-Shot (No LoRA)", True)]:
    s_em, s_mf1, s_tf1, s_t = evaluate(test_seen_df, three_shot=is_3shot)
    u_em, u_mf1, u_tf1, u_t = evaluate(test_unseen_df, three_shot=is_3shot)
    print("\n" + "="*58)
    print(f"Un-Tuned Llama-3.2-1B-Instruct — {mode_name}")
    print("="*58)
    print(f"Inference Latency:         {(s_t/len(test_seen_df))*1000:.2f} ms/sample")
    print(f"Seen Ambiguous Exact Match:{s_em*100:.2f}% | Macro F1: {s_mf1*100:.2f}% | Token F1: {s_tf1*100:.2f}%")
    print(f"Unseen Zero-Shot EM:       {u_em*100:.2f}% | Macro F1: {u_mf1*100:.2f}% | Token F1: {u_tf1*100:.2f}%")

print(f"\nPeak GPU VRAM: {torch.cuda.max_memory_allocated() / 1024**2:.1f} MB")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and 


Un-Tuned Llama-3.2-1B-Instruct — Zero-Shot (No LoRA)
Inference Latency:         97.37 ms/sample
Seen Ambiguous Exact Match:0.60% | Macro F1: 0.27% | Token F1: 4.72%
Unseen Zero-Shot EM:       1.20% | Macro F1: 0.17% | Token F1: 4.70%


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug


Un-Tuned Llama-3.2-1B-Instruct — Three-Shot (No LoRA)
Inference Latency:         199.11 ms/sample
Seen Ambiguous Exact Match:1.40% | Macro F1: 0.56% | Token F1: 5.70%
Unseen Zero-Shot EM:       1.60% | Macro F1: 0.34% | Token F1: 5.98%

Peak GPU VRAM: 5211.0 MB
